# 01 — Exploratory Data Analysis
**Congestion Pricing Impact Analyzer**

Policy shock: NYC MTA Congestion Relief Zone → **2025-01-05**

This notebook explores the raw trip data and builds intuition before causal estimation:
- Volume trends: CBD vs non-CBD
- Parallel pre-trends (visual)
- Fare / fee composition shifts
- Borough-level substitution patterns
- Zone-level heatmaps

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
SHOCK = pd.Timestamp('2025-01-05')
PROC  = Path('../artifacts/processed')
print('Ready.')

## 1. Load enriched panel

In [ ]:
panel_path = PROC / 'zone_day_panel_enriched.parquet'
if not panel_path.exists():
    panel_path = PROC / 'zone_day_panel.parquet'

panel = pd.read_parquet(panel_path)
panel['date'] = pd.to_datetime(panel['date'])
print(f'Panel shape: {panel.shape}')
print(f'Date range:  {panel.date.min().date()} → {panel.date.max().date()}')
print(f'Zones:       {panel.zone_id.nunique()}')
panel.head()

## 2. Data completeness & missing cells

In [ ]:
n_dates = panel['date'].nunique()
n_zones = panel['zone_id'].nunique()
completeness = len(panel) / (n_dates * n_zones) * 100
print(f'Total dates:  {n_dates}')
print(f'Total zones:  {n_zones}')
print(f'Max cells:    {n_dates * n_zones:,}')
print(f'Actual rows:  {len(panel):,}')
print(f'Completeness: {completeness:.1f}%')

# Missing-by-date heatmap (sample of zones)
sample_zones = panel['zone_id'].value_counts().head(30).index
pivot = panel[panel['zone_id'].isin(sample_zones)].pivot_table(
    index='zone_id', columns=pd.Grouper(key='date', freq='W'),
    values='trip_count', aggfunc='sum'
)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot.isnull(), ax=ax, cmap='Reds', cbar=False, xticklabels=False)
ax.set_title('Missing cells (red) — sample of 30 zones', fontsize=12)
plt.tight_layout(); plt.show()

## 3. Aggregate daily volumes: CBD vs Non-CBD

In [ ]:
daily = (
    panel.groupby(['date', 'treat_binary'])['trip_count']
    .sum().reset_index()
    .rename(columns={'treat_binary': 'group'})
)
daily['group'] = daily['group'].map({1: 'CBD (treated)', 0: 'Non-CBD (control)'})
daily['rolling7'] = daily.groupby('group')['trip_count'].transform(
    lambda s: s.rolling(7, min_periods=3).mean()
)

fig = px.line(
    daily, x='date', y='rolling7', color='group',
    color_discrete_map={'CBD (treated)': '#2563EB', 'Non-CBD (control)': '#9CA3AF'},
    labels={'rolling7': '7-day rolling avg trips', 'date': ''},
    title='Daily Trip Volume (7-day MA): CBD vs Non-CBD'
)
fig.add_vline(x='2025-01-05', line_dash='dash', line_color='red',
              annotation_text='Congestion Pricing')
fig.update_layout(height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

## 4. Pre-trend visual check

In [ ]:
# Normalise to 100 at the week before shock for parallel-trends visual
weekly = (
    panel.groupby([pd.Grouper(key='date', freq='W'), 'treat_binary'])['trip_count']
    .sum().reset_index()
)
weekly.rename(columns={'treat_binary': 'group', 'date': 'week'}, inplace=True)
weekly['group'] = weekly['group'].map({1: 'CBD', 0: 'Non-CBD'})

# Index to week before shock
ref_week = weekly[weekly['week'] < SHOCK]['week'].max()
ref_vals = weekly[weekly['week'] == ref_week].set_index('group')['trip_count']
weekly['indexed'] = weekly.apply(
    lambda r: r['trip_count'] / ref_vals[r['group']] * 100, axis=1
)

fig, ax = plt.subplots(figsize=(13, 5))
for grp, color in [('CBD', '#2563EB'), ('Non-CBD', '#9CA3AF')]:
    sub = weekly[weekly['group'] == grp]
    pre  = sub[sub['week'] <= SHOCK]
    post = sub[sub['week'] > SHOCK]
    ax.plot(pre['week'], pre['indexed'], color=color, linewidth=2, label=grp)
    ax.plot(post['week'], post['indexed'], color=color, linewidth=2, linestyle='--')

ax.axvline(SHOCK, color='red', linewidth=1.8, linestyle='--', alpha=0.8)
ax.axhline(100, color='black', linewidth=0.7, linestyle=':')
ax.set_xlabel('Week'); ax.set_ylabel('Trip index (100 = week before shock)')
ax.set_title('Pre-Trend Visual Check: Indexed Weekly Trips', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print("If pre-period lines are parallel → parallel trends assumption plausible.")

## 5. Fare and fee composition

In [ ]:
# Compare avg_fare and avg_cbd_fee before vs after shock
comp = panel.copy()
comp['period'] = np.where(comp['date'] < SHOCK, 'Pre', 'Post')

fare_comp = (
    comp.groupby(['period', 'treat_binary'])[['avg_fare', 'avg_cbd_fee', 'surcharge_sum']]
    .mean().reset_index()
)
fare_comp['treat_binary'] = fare_comp['treat_binary'].map({1: 'CBD', 0: 'Non-CBD'})

fig = make_subplots(rows=1, cols=3,
    subplot_titles=['Avg Fare ($)', 'Avg CBD Fee ($)', 'Avg Surcharge ($)'])

for i, col in enumerate(['avg_fare', 'avg_cbd_fee', 'surcharge_sum'], 1):
    for grp, color in [('CBD', '#2563EB'), ('Non-CBD', '#9CA3AF')]:
        sub = fare_comp[fare_comp['treat_binary'] == grp]
        fig.add_trace(go.Bar(
            x=sub['period'], y=sub[col], name=grp,
            marker_color=color, showlegend=(i == 1)
        ), row=1, col=i)

fig.update_layout(barmode='group', height=380, title_text='Fare Composition: Pre vs Post',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

## 6. Borough-level substitution analysis

In [ ]:
if 'borough' in panel.columns:
    boro = (
        panel.assign(period=np.where(panel['date'] < SHOCK, 'Pre', 'Post'))
        .groupby(['period', 'borough'])['trip_count']
        .sum().reset_index()
    )
    boro_pivot = boro.pivot(index='borough', columns='period', values='trip_count')
    boro_pivot['pct_change'] = (boro_pivot['Post'] / boro_pivot['Pre'] - 1) * 100
    boro_pivot = boro_pivot.sort_values('pct_change')

    fig, ax = plt.subplots(figsize=(8, 5))
    colors = ['#DC2626' if v < 0 else '#16A34A' for v in boro_pivot['pct_change']]
    ax.barh(boro_pivot.index, boro_pivot['pct_change'], color=colors, alpha=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Trip count % change (post vs pre)', fontsize=11)
    ax.set_title('Borough-Level Trip Change After Congestion Pricing', fontsize=12, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print("Borough column not in panel. Run step 3 (enrich_panel) first.")

## 7. Peak vs off-peak effects

In [ ]:
panel['period'] = np.where(panel['date'] < SHOCK, 'Pre', 'Post')

if 'peak_share' in panel.columns:
    peak_comp = (
        panel[panel['treat_binary'] == 1]  # CBD zones only
        .assign(peak_bin=pd.cut(panel['peak_share'], bins=[0, 0.33, 0.66, 1],
                                labels=['Low peak', 'Mid peak', 'High peak']))
        .groupby(['period', 'peak_bin'])['trip_count']
        .mean().reset_index()
    )

    fig = px.bar(
        peak_comp, x='peak_bin', y='trip_count', color='period', barmode='group',
        color_discrete_map={'Pre': '#9CA3AF', 'Post': '#2563EB'},
        title='CBD Trips by Peak Share Bin: Pre vs Post',
        labels={'trip_count': 'Avg daily trips', 'peak_bin': 'Peak hour intensity'}
    )
    fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

## 8. CBD fee distribution (post-period only)

In [ ]:
if 'avg_cbd_fee' in panel.columns:
    post_cbd = panel[(panel['date'] >= SHOCK) & (panel['treat_binary'] == 1) 
                     & (panel['avg_cbd_fee'] > 0)]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(post_cbd['avg_cbd_fee'], bins=40, color='#EA580C', alpha=0.75, edgecolor='white')
    ax.axvline(post_cbd['avg_cbd_fee'].mean(), color='black', linewidth=1.5,
               linestyle='--', label=f"Mean = ${post_cbd['avg_cbd_fee'].mean():.2f}")
    ax.set_xlabel('Avg CBD Congestion Fee per zone-day ($)', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title('Distribution of CBD Congestion Fees (Post-Period, Treated Zones)', fontsize=12)
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"This is our continuous 'dose' variable for L3 Continuous DiD.")

## 9. Summary statistics table

In [ ]:
cols = ['trip_count', 'avg_distance', 'avg_duration', 'avg_fare', 'avg_cbd_fee', 'peak_share']
cols = [c for c in cols if c in panel.columns]

summary = (
    panel.assign(period=np.where(panel['date'] < SHOCK, 'Pre', 'Post'))
    .groupby(['period', 'treat_binary'])[cols]
    .mean()
    .round(3)
)
summary.index = summary.index.map(lambda x: f"{x[0]} / {'CBD' if x[1]==1 else 'Non-CBD'}")
print("Mean values by period × treatment group:")
summary